In [ ]:
from enum import Enum
import cvxpy as cp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.ticker import MaxNLocator
import pandas as pd

class Season(Enum):
	WINTER = 'jan'
	SPRING = 'apr'
	SUMMER = 'jul'
	AUTUMN = 'oct'

DAYS = 30					# number of days to model
I = 1						# number of containers to model
season = Season.SUMMER		# season for RTM prices

# !!! OPERATIONAL PARAMETERS - DO NOT CHANGE !!!
DELTA_t = 1.0		# hr
HOURS_PER_DAY = 24
N = DAYS * HOURS_PER_DAY
SOC_MIN = 0.05		# dimensionless
SOC_MAX = 0.95		# dimensionless
Q_NEW = 3.916		# MWh, new container capacity
P_MAX = 0.979		# MW, hard limit
T_MIN = -30			# °C
T_REF = 25			# °C
T_MAX = 50			# °C
ETA = 0.968			# dimensionless

# FLEXIBLE MODEL PARAMS - CAN CHANGE FOR SENSITIVITY ANALYSIS
ALPHA = 2.61e-5		# MWh capacity lost per MWh throughput
BETA = 2.5			# °C / MWh throughput
GAMMA = 3.0			# °C / hr
KAPPA = 0.04		# / °C
RHO = 19500			# $ / MWh capacity lost (opportunity)
SIGMA = 200000		# $ / MWh capacity lost (replacement)
A = 0.02			# dimensionless
B = 0.02			# dimensionless

# Day starting values for containers, spaced equally in range
SOH_0 = np.linspace(0.98, 0.82, I)	# dimensionless
SOC_0 = np.linspace(0.8, 0.2, I)	# dimensionless

# Day ending values
SOC_N = SOC_0

expected_prices_df = pd.read_csv("./data/raw/DAM/seasonal_stats.csv")
LAMBDA_DAILY = expected_prices_df[f'{season.value}_mean'].to_numpy().reshape(-1, 1)
LAMBDA = np.tile(LAMBDA_DAILY, (DAYS, 1))

def solve_bess_scenario(aging=True):

	# states: k = 0 -> N, i = 1 -> 2
	E = cp.Variable((N + 1, I), nonneg=True)
	Q = cp.Variable((N + 1, I), nonneg=True)
	T = cp.Variable((N + 1, I), nonneg=True)

	# decisions: k = 0 -> N-1, i = 1 -> 2
	c = cp.Variable((N, I), nonneg=True)
	d = cp.Variable((N, I), nonneg=True)
	u = cp.Variable((N, I), nonneg=True)

	phi = cp.Parameter((N,I), nonneg=True)
	delta_Q = ALPHA * cp.multiply(phi, c + d) * DELTA_t

	constraints = [
		# Initial conditions
		Q[0, :] == Q_NEW * SOH_0,
		E[0, :] == cp.multiply(Q[0, :], SOC_0),
		T[0, :] == T_REF,
		# Dynamics - see below for capacity dynamics
		Q[1:, :] == Q[:-1, :] - delta_Q,
		E[1:, :] == E[:-1, :] + (ETA * c * DELTA_t) - (d * DELTA_t / ETA),
		T[1:, :] == T[:-1, :] + (BETA * (c + d) * DELTA_t) - (GAMMA * u * DELTA_t),
		# Boundary conditions
		# Q >= 0, # variable created as non-neg=True
		E >= Q * SOC_MIN,
		E <= Q * SOC_MAX,
		T >= T_MIN,
		T <= T_MAX,
		# c >= 0, # variable created as non-neg=True
		c <= P_MAX,
		# d >= 0, # variable created as non-neg=True
		d <= P_MAX,
		# u >= 0, # variable created as non-neg=True
		u <= 1,
		# Terminal conditions
		E[N, :] >= cp.multiply(Q[N, :], SOC_N)
	]

	arbitrage_cost = cp.sum(cp.multiply(LAMBDA, c - d) * DELTA_t)
	operating_cost = cp.sum(cp.multiply(LAMBDA, A + B * u) * P_MAX * DELTA_t)
	opportunity_cost = cp.sum(RHO * delta_Q)
	replacement_cost = cp.sum(SIGMA * delta_Q)
	if (aging):
		objective = cp.Minimize(arbitrage_cost + operating_cost + opportunity_cost + replacement_cost)
	else:
		objective = cp.Minimize(arbitrage_cost + operating_cost)
	problem = cp.Problem(objective, constraints)

	T_est = np.full((N, I), T_REF)

	for iteration in range(10):
		print(f"Iteration {iteration} starting...")

		phi.value = 1 + KAPPA * np.maximum(0, T_est - T_REF)

		problem.solve(
			solver=cp.ECOS,
			feastol=1e-8
		)

		if problem.status == "optimal":
			T_est = T.value[:-1, :]
			print(f"Iteration {iteration}: Total Cost = {problem.value:.2f}")
		else:
			print("Solver failed to find an optimal solution.")
			break

	gross_revenue = cp.sum(cp.multiply(LAMBDA, d) * DELTA_t).value
	charging_expenditure = cp.sum(cp.multiply(LAMBDA, c) * DELTA_t).value
	op_cost_val = operating_cost.value
	opport_cost_val = opportunity_cost.value
	replac_cost_val = replacement_cost.value

	results = {
		'revenue': gross_revenue,
		'charge_cost': charging_expenditure,
		'op_cost': op_cost_val,
		'opportunity_cost': opport_cost_val,
		'replacement_cost': replac_cost_val,
		'net_profit': gross_revenue - (charging_expenditure + op_cost_val + opport_cost_val + replac_cost_val),
		'capacity_loss': (Q.value[0,0] - Q.value[-1,0]),
		'energy': E.value,  # Shape (N+1, I)
		'dispatch': c.value - d.value,
		'temp': T.value
	}
	return results

aware = solve_bess_scenario(aging=True)
agnostic = solve_bess_scenario(aging=False)

# Header
print(f"{'Financial Metric':<35} | {'Aging-Aware':<15} | {'Aging-Agnostic':<15}")
print("-" * 72)

# Row definitions
metrics = [
	("Gross Discharge Revenue ($)", 'revenue'),
	("Gross Charging Cost ($)", 'charge_cost'),
	("Operating (Parasitic) Cost ($)", 'op_cost'),
	("Opportunity Cost (Degradation) ($)", 'opportunity_cost'),
	("Replacement Cost (Degradation) ($)", 'replacement_cost'),
]

for label, key in metrics:
	print(f"{label:<35} | {aware[key]:<15.2f} | {agnostic[key]:<15.2f}")

print("-" * 72)
print(f"{'TOTAL NET PROFIT ($)':<35} | {aware['net_profit']:<15.2f} | {agnostic['net_profit']:<15.2f}")
print("-" * 72)
print(f"{'Total Capacity Lost (MWh)':<35} | {aware['capacity_loss']:<15.6f} | {agnostic['capacity_loss']:<15.6f}")


In [ ]:
target_day = 28
start_idx = (target_day - 1) * 24
end_idx = target_day * 24

e_aware_day = aware['energy'][start_idx : end_idx + 1, 0]
e_agnostic_day = agnostic['energy'][start_idx : end_idx + 1, 0]

lambda_day = LAMBDA[start_idx : end_idx].flatten()

x_hours = np.arange(25)
x_intervals = np.arange(24)

fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.plot(x_hours, e_aware_day,
	label='Aging-Aware',
	color='#2ca02c', linewidth=2.5, marker='o', markersize=4)

ax1.plot(x_hours, e_agnostic_day,
	label='Aging-Agnostic',
	color='#d62728', linewidth=2, linestyle='--', marker='x', markersize=4)

ax1.set_xlabel('Hour of Day', fontsize=12)
ax1.set_ylabel('Stored Energy $E$ (MWh)', fontsize=12, fontweight='bold')
ax1.set_ylim(0, Q_NEW + 0.2)
ax1.set_xticks(np.arange(0, 25, 1))
ax1.grid(True, linestyle='--', alpha=0.5)

ax2 = ax1.twinx()
ax2.step(x_intervals, lambda_day,
	where='post', color='black', alpha=0.15, linewidth=1.5,
	label='Market Price ($\lambda$)')

ax2.set_ylabel('RTM Price ($/MWh)', fontsize=12, fontweight='bold', alpha=0.6)

plt.title(f'Aging-Aware vs. Baseline Controller - Stored Energy (Summer)', fontsize=14)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
labels = ['Ignoring Degradation', 'Including Degradation']
aware_values = [45727.26, 29603.87]
agnostic_values = [47502.65, 23028.31]

x = np.arange(len(labels))
width = 0.25

fig, ax = plt.subplots(figsize=(6, 8))
rects1 = ax.bar(x - width/2 - 0.02, aware_values, width, label='Aging-Aware', color='#2ca02c')
rects2 = ax.bar(x + width/2 + 0.02, agnostic_values, width, label='Aging-Agnostic', color='#d62728')

label_fontsize = 14
tick_fontsize = 12
annotation_fontsize = 11

ax.set_ylabel('Net Profit (USD)', fontweight='bold', fontsize=label_fontsize)
ax.set_title('Annualized Profit for One Container:\nAging-Aware vs. Aging-Agnostic', fontweight='bold', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=tick_fontsize)
ax.tick_params(axis='y', labelsize=tick_fontsize)

# Revert legend to top right
ax.legend(fontsize=tick_fontsize, loc='upper right')

def autolabel(rects):
	for rect in rects:
		height = rect.get_height()
		ax.annotate(f'${height:,.2f}',
			xy=(rect.get_x() + rect.get_width() / 2, height),
			xytext=(0, 5),
			textcoords="offset points",
			ha='center', va='bottom', fontsize=annotation_fontsize, fontweight='bold')

autolabel(rects1)
autolabel(rects2)

ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('profit_comparison_portrait_topright.png')